# 04 - Gold
Builds the **gold-layer Delta tables** that back the Direct Lake *Fabric Arch Review - Governance* semantic model and report. Reads `findings.json` + `raw/` from the same `RUN_ID` folder and **appends** one partition of history per run to the Lakehouse `Tables/`. No Azure/Fabric admin auth needed - this stage is offline + Spark only.

## Parameters

In [ ]:
# Parameters (injected by the pipeline)
GITHUB_REPO_URL = "https://github.com/biro98/fabric-architecture-review.git"
GITHUB_BRANCH = "main"
GITHUB_REF = ""  # pinned release ref from the pipeline (blank = GITHUB_BRANCH tip)
CLIENT_NAME = "Contoso"
ENGAGEMENT_NAME = "Fabric Architecture Review"
REVIEWER_NAME = ""
RUN_ID = "latest"

## Clone framework + install dependencies

In [ ]:
import os, sys, shutil, subprocess
WORK_ROOT = "/tmp/fabric-arch-review-run"
REPO_DIR = os.path.join(WORK_ROOT, "repo")
os.makedirs(WORK_ROOT, exist_ok=True)
_url = GITHUB_REPO_URL
_ref = (GITHUB_REF or "").strip() or GITHUB_BRANCH
def _git(args, **kw):
    r = subprocess.run(["git"] + args, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        out = ((r.stderr or "") + (r.stdout or "")).strip()
        raise RuntimeError("git " + " ".join(args) + " failed (rc=" + str(r.returncode) + "): " + out)
    return r

subprocess.run(["git", "config", "--global", "--add", "safe.directory", REPO_DIR])

def _fresh_clone():
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    _git(["clone", "--branch", _ref, "--depth", "1", _url, REPO_DIR])

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    try:
        _git(["-C", REPO_DIR, "remote", "set-url", "origin", _url])
        _git(["-C", REPO_DIR, "fetch", "--depth", "1", "origin", _ref])
        _git(["-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"])
    except RuntimeError:
        _fresh_clone()
else:
    _fresh_clone()
del _url
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
_pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")])
if _pip.returncode != 0:
    print("WARNING: pip install exited", _pip.returncode, "- a benign dependency-resolver warning is expected on Fabric; continuing.")
print("Repo + deps ready at", REPO_DIR)

## Build + append gold Delta tables

In [ ]:
import os, datetime as dt
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                               DoubleType, BooleanType, TimestampType)
from reports.gold_layer import build_gold_from_dir
from reports.powerbi.schema import GOLD_TABLES_BY_NAME

LH = "/lakehouse/default/Files"
_BASE = os.path.join(LH, "fabric-arch-review")
if RUN_ID == "latest" and os.path.isdir(_BASE):
    _runs = [d for d in os.listdir(_BASE) if os.path.isdir(os.path.join(_BASE, d, "raw"))]
    if _runs:
        RUN_ID = max(_runs, key=lambda d: os.path.getmtime(os.path.join(_BASE, d)))
        print("Resolved RUN_ID=latest ->", RUN_ID)
OUT_DIR = os.path.join(LH, "fabric-arch-review", RUN_ID)
if not os.path.isdir(os.path.join(OUT_DIR, "raw")):
    raise RuntimeError("Run folder not found: " + OUT_DIR + " -- run Collect + Analyze first.")

RUN_TS = dt.datetime.now(dt.timezone.utc).replace(microsecond=0, tzinfo=None).isoformat() + "Z"

spark = SparkSession.builder.getOrCreate()
# Read the DEPLOYED FAR version recorded by setup.ipynb (meta_deployment table).
# build_gold compares it to the latest published release to drive the report's
# version banner. Falls back to the running code's VERSION file when absent.
_deployed_version = None
try:
    _mrows = spark.table("meta_deployment").select("version").collect()
    if _mrows:
        _deployed_version = _mrows[0]["version"]
        print("  deployed FAR version:", _deployed_version)
except Exception as _e:
    print("  meta_deployment not found (using code VERSION):", _e)
tables = build_gold_from_dir(
    OUT_DIR, run_id=RUN_ID, run_timestamp=RUN_TS,
    client_name=CLIENT_NAME, engagement_name=ENGAGEMENT_NAME, reviewer_name=REVIEWER_NAME,
    deployed_version=_deployed_version, repo_url=GITHUB_REPO_URL, branch=GITHUB_BRANCH,
)

spark = SparkSession.builder.getOrCreate()
_TYPE = {"string": StringType(), "int64": LongType(), "double": DoubleType(),
         "boolean": BooleanType(), "dateTime": TimestampType()}

def _schema(table):
    return StructType([StructField(c.name, _TYPE[c.kind], True) for c in table.columns])

def _rows(rows, table):
    cols = [(c.name, c.kind) for c in table.columns]
    out = []
    for r in rows:
        rec = []
        for name, kind in cols:
            v = r.get(name)
            if kind == "dateTime" and isinstance(v, str) and v:
                v = dt.datetime.fromisoformat(v.replace("Z", "+00:00")).replace(tzinfo=None)
            rec.append(v)
        out.append(tuple(rec))
    return out

print("Gold build for run", RUN_ID, "@", RUN_TS)
for name, rows in tables.items():
    table = GOLD_TABLES_BY_NAME[name]
    schema = _schema(table)
    df = spark.createDataFrame(_rows(rows, table), schema=schema)
    (df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(name))
    print("  " + name + ": appended " + str(len(rows)) + " row(s)")

# Flag only the newest run as is_latest so the Direct Lake report defaults to it.
try:
    from pyspark.sql import functions as F
    from delta.tables import DeltaTable
    _max_ts = spark.table("gold_run_summary").agg(F.max("run_timestamp").alias("m")).collect()[0]["m"]
    DeltaTable.forName(spark, "gold_run_summary").update(set={"is_latest": F.col("run_timestamp") == F.lit(_max_ts)})
    print("  gold_run_summary: is_latest set for newest run @", _max_ts)
except Exception as _e:
    print("  is_latest update skipped:", _e)

try:
    import notebookutils
    notebookutils.notebook.exit("gold:" + OUT_DIR)
except Exception:
    pass